In [ ]:
from partial_discharge_adaptive_fusion.protocol import assert_protocol_frozen, load_experiment_config
CONFIG = load_experiment_config('configs/experiments/two-dataset-confirmatory-v2-batch4-localraw.yaml')
assert_protocol_frozen(CONFIG)
assert CONFIG['experts']['batch_size'] == CONFIG['experts']['inference_batch_size'] == 4
print('Using config:', CONFIG['config_version'])

# Reliability-aware adaptive fusion

**Objective.** Estimate sample-wise temporal/CWT correctness from leakage-free OOF rows and evaluate reliability-proportional fusion.

**Inputs.** Independent per-seed v2-batch4 OOF predictions/features and validation predictions.

**Outputs.** Reliability models, quality tables, sample-wise weights and adaptive predictions.

**Experimental role.** Primary adaptive method.

**Leakage constraints.** Correctness targets come only from expert predictions made without training on that OOF sample; all estimator and threshold choices use validation only; reject mixed config versions.

In [ ]:
from partial_discharge_adaptive_fusion.fusion import (
    adaptive_probability, fit_reliability_models, predict_reliability,
)

print('Reliability candidates:', ['logistic', 'hist_gradient_boosting', 'small_mlp'])

def apply_adaptive(oof_features, oof_labels, oof_temporal, oof_cwt, evaluation_features, evaluation_temporal, evaluation_cwt, model_name, seed):
    models = fit_reliability_models(
        oof_features, oof_labels, oof_temporal, oof_cwt, model_name=model_name, seed=seed,
    )
    temporal_reliability, cwt_reliability = predict_reliability(models, evaluation_features)
    probability, weights = adaptive_probability(
        evaluation_temporal, evaluation_cwt, temporal_reliability, cwt_reliability,
    )
    return models, probability, weights

print('Adaptive output uses reliability-proportional sample-wise weights trained from OOF only.')

## Findings and handoff

Reliability AUC or calibration alone does not establish useful fusion; the primary endpoint is adaptive versus best fixed MCC on locked data.

**Next stage:** apply the conservative fallback rule without changing the mechanism per dataset.